# 0. Goals

- Karena ini klasifikasi maka baseline model nya menggunakan Decision Tree
- Mencari model terbaik dengan kondisi data se adanya menggunakan Pycaret
- Melakukan Hyperparameter Tuning

# 1. Import Library, Data, Pastikan Data Berhasil di Import

In [48]:
import pandas as pd
import numpy as np
import optuna
import time
import shap
import joblib

from pycaret.tasks import ClassificationExperiment

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.feature_selection import mutual_info_classif
from catboost import CatBoostClassifier

In [49]:
path_df = r'../../Data/result/02_Data_Cleaning.csv'

In [50]:
df_fraud = pd.read_csv(path_df)

In [51]:
df_fraud.head(2)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0


# 2. Split data sebelum masuk ke PyCaret

In [52]:
X = df_fraud.drop(columns='Class')
y = df_fraud['Class']

In [53]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [54]:
train = X_train.copy()
train['Class'] = y_train

test = X_test.copy()
test['Class'] = y_test

# 3. Setup PyCaret

In [55]:
# exp = ClassificationExperiment(target='Class', session_id=42)

In [56]:
# exp.fit(train)

In [57]:
# exp.add_metric(
#     id='auprc',
#     name='AUPRC',
#     score_func=average_precision_score,
#     target='pred_proba',
#     greater_is_better=True
# )

# 4. Tes Model Decision Tree

In [58]:
# dt = exp.create_model('dt')

In [59]:
# dt

Untuk model Decision Tree, Score Base untuk AUPRC nya adalah 0.5450

# 5. Compare Model

In [60]:
# best = exp.compare_models(sort='auprc')

In [61]:
# model_names = {
#     'catboost': 'CatBoost Classifier',
#     'et': 'Extra Trees Classifier',
#     'rf': 'Random Forest Classifier',
#     'ada': 'AdaBoost Classifier',
#     'dt': 'Decision Tree Classifier',
#     'lr': 'Logistic Regression',
#     'lda': 'Linear Discriminant Analysis',
#     'lightgbm': 'LightGBM Classifier',
#     'xgboost': 'XGBoost Classifier',
#     'knn': 'K-Nearest Neighbors',
#     'nb': 'Naive Bayes',
#     'svm': 'Support Vector Machine',
#     'ridge': 'Ridge Classifier',
#     'qda': 'Quadratic Discriminant Analysis',
#     'gbc': 'Gradient Boosting Classifier',
#     'dummy': 'Dummy Classifier'
# }

In [62]:
# results = (
#     best.leaderboard
#     .sort_values(by='AUPRC', ascending=False)
#     .reset_index(drop=True)
# )

# results['Model'] = results['Model'].map(model_names)

# results

Kesimpulan :

Setelah dilakukan comparing, ditemukan bahwa dalam kondisi data minim cleaning, top 3 best model yang melebihi baseline adalah
- Extra Trees Classifier dengan AUPRC 0.8451
- Random Forest Classifier dengan AUPRC 0.8261
- Adaboost Classifier dengan AUPRC 0.7760

# 6. Hyperparameter Tuning Top 3 Model

## 6.1 Extra Trees Classifier

In [63]:
# def search_best_params_extra_trees_classifier(trial):

#     max_features_type = trial.suggest_categorical(
#         "max_features_type",
#         ["sqrt", "log2", "float"]
#     )

#     if max_features_type == "float":
#         max_features = trial.suggest_float(
#             "max_features_float",
#             0.3,
#             1.0
#         )
#     else:
#         max_features = max_features_type
        
#     params ={
#         "n_estimators"      : trial.suggest_int("n_estimators", 100, 1000),                                        # --> jumlah pohon
#         "max_depth"         : trial.suggest_int("max_depth", 2, 50),                                               # --> Maks Kedalaman
#         "min_samples_split" : trial.suggest_int("min_samples_split", 2, 50),                                       # --> Min sample untuk di split
#         "min_samples_leaf"  : trial.suggest_int("min_samples_leaf", 1, 20),                                        # --> Min sample pada daun
#         "max_features"      : max_features,                                                                        # --> Min sample pada daun
#         "criterion"         : trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),             # --> Cara ukur kualitas split
#         "class_weight"      : trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]), # --> Kasih bobot ke target
#         "random_state"      : 42,
#         "n_jobs"            : -1
#     }

#     model = ExtraTreesClassifier(**params)

#     cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    
#     scores = []

#     for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):

#         X_tr = X_train.iloc[train_idx]
#         X_val = X_train.iloc[valid_idx]

#         y_tr = y_train.iloc[train_idx]
#         y_val = y_train.iloc[valid_idx]

#         model.fit(X_tr, y_tr)

#         y_proba = model.predict_proba(X_val)[:, 1]

#         auprc = average_precision_score(
#             y_val,
#             y_proba
#         )

#         scores.append(auprc)

#         # report intermediate result
#         trial.report(
#             np.mean(scores),
#             step=fold
#         )

#     return np.mean(scores)

In [64]:
# study = optuna.create_study(direction="maximize")

# study.optimize(search_best_params_extra_trees_classifier, n_trials=25)

In [65]:
# print("Best AUPRC Extra Trees Classifier:", study.best_value)
# print("Best Extra Trees Classifier Parameters:")
# print(study.best_params)

## 6.2 Random Forest Classifier

In [66]:
# def search_best_params_random_forest_classifier(trial):

#     max_features_type = trial.suggest_categorical(
#         "max_features_type",
#         ["sqrt", "log2", "float"]
#     )

#     if max_features_type == "float":
#         max_features = trial.suggest_float(
#             "max_features_float",
#             0.3,
#             1.0
#         )
#     else:
#         max_features = max_features_type
        
#     params ={
#         "n_estimators"      : trial.suggest_int("n_estimators", 100, 1000),                                        # --> jumlah pohon
#         "max_depth"         : trial.suggest_int("max_depth", 2, 50),                                               # --> Maks Kedalaman
#         "min_samples_split" : trial.suggest_int("min_samples_split", 2, 50),                                       # --> Min sample untuk di split
#         "min_samples_leaf"  : trial.suggest_int("min_samples_leaf", 1, 20),                                        # --> Min sample pada daun
#         "max_features"      : max_features,                                                                        # --> Min sample pada daun
#         "criterion"         : trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),             # --> Cara ukur kualitas split
#         "class_weight"      : trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]), # --> Kasih bobot ke target
#         "bootstrap"         : trial.suggest_categorical("bootstrap", [True, False]),                               # --> 
#         "random_state"      : 42,
#         "n_jobs"            : -1
#     }

#     model = RandomForestClassifier(**params)

#     cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    
#     scores = []

#     for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):

#         X_tr = X_train.iloc[train_idx]
#         X_val = X_train.iloc[valid_idx]

#         y_tr = y_train.iloc[train_idx]
#         y_val = y_train.iloc[valid_idx]

#         model.fit(X_tr, y_tr)

#         y_proba = model.predict_proba(X_val)[:, 1]

#         auprc = average_precision_score(
#             y_val,
#             y_proba
#         )

#         scores.append(auprc)

#         # report intermediate result
#         trial.report(
#             np.mean(scores),
#             step=fold
#         )

#     return np.mean(scores)

In [67]:
# rf_study = optuna.create_study(direction="maximize")

# rf_study.optimize(search_best_params_random_forest_classifier, n_trials=25)

In [68]:
# print("Best AUPRC Random Forest Classifier:", rf_study.best_value)
# print("Best Random Forest Classifier Parameters:")
# print(rf_study.best_params)

## 6.3 CatBoost Classifier

In [69]:
# def search_best_params_catboost_classifier(trial):
#     params = {
#         "iterations"        : trial.suggest_int("iterations", 100, 1000),
#         "depth"             : trial.suggest_int("depth", 4, 10),
#         "learning_rate"     : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
#         "l2_leaf_reg"       : trial.suggest_float("l2_leaf_reg", 1, 10, log=True),
#         "random_strength"   : trial.suggest_float("random_strength", 0, 2),
#         "loss_function"     : "Logloss",
#         "eval_metric"       : "PRAUC",
#         "random_seed"       : 42,
#         "verbose"           : False,
#         "thread_count"      : -1,
#     }

#     model = CatBoostClassifier(**params)

#     cv = StratifiedKFold(
#         n_splits=5,
#         shuffle=True,
#         random_state=42
#     )

#     scores = []

#     for fold, (train_idx, valid_idx) in enumerate(
#         cv.split(X_train, y_train)
#     ):

#         X_tr = X_train.iloc[train_idx]
#         X_val = X_train.iloc[valid_idx]

#         y_tr = y_train.iloc[train_idx]
#         y_val = y_train.iloc[valid_idx]

#         model.fit(X_tr, y_tr)

#         y_proba = model.predict_proba(X_val)[:, 1]

#         auprc = average_precision_score(
#             y_val,
#             y_proba
#         )

#         scores.append(auprc)

#         # report intermediate result
#         trial.report(
#             np.mean(scores),
#             step=fold
#         )

#         if trial.should_prune():
#             raise optuna.TrialPruned()

#     return np.mean(scores)

In [70]:
# catboost_study = optuna.create_study(direction="maximize")

# catboost_study.optimize(search_best_params_catboost_classifier, n_trials=15)

In [71]:
# print("Best AUPRC Catboost Classifier:", catboost_study.best_value)
# print("Best Catboost Classifier Parameters:")
# print(catboost_study.best_params)

# SUMMARY OPTUNA

In [72]:
# Dengan AUPRC 0.849
best_ETC_params = {
    'n_estimators'        : 296, 
    'max_depth'           : 37, 
    'min_samples_split'   : 11, 
    'min_samples_leaf'    : 3, 
    'criterion'           : 'entropy', 
    'class_weight'        : None
}

# Dengan AUPRC 0.852
best_RFC_params = {
    'n_estimators'        : 691, 
    'max_depth'           : 23, 
    'min_samples_split'   : 4, 
    'min_samples_leaf'    : 4, 
    'criterion'           : 'log_loss', 
    'class_weight'        : 'balanced_subsample', 
    'bootstrap'           : False
}

# Dengan AUPRC 0.853
best_CB_params = {
    'iterations': 594, 
    'depth': 9, 
    'learning_rate': 0.010639915618971553, 
    'l2_leaf_reg': 1.1199438541669806, 
    'random_strength': 0.8875933831253247
}

# Train Model

In [73]:
etc_model = ExtraTreesClassifier(
    **best_ETC_params,
    random_state=42,
    n_jobs=-1
)

rfc_model = RandomForestClassifier(
    **best_RFC_params,
    random_state=42,
    n_jobs=-1
)

cb_model = CatBoostClassifier(
    **best_CB_params,
    verbose=False,
    thread_count=-1
)

## Extra Trees Classifier

In [74]:
start_etc_train = time.perf_counter()

etc_model.fit(X_train, y_train)

end_etc_train = time.perf_counter()

training_etc_model_time = end_etc_train - start_etc_train

In [75]:
start_etc_test = time.perf_counter()

etc_pred = etc_model.predict_proba(X_test)[:, 1]

end_etc_test = time.perf_counter()

test_etc_model_time = end_etc_test - start_etc_test

In [76]:
etc_v1_auprc = average_precision_score(y_test, etc_pred)

In [78]:
print(f"Training time Extra Trees Classifier Model \t: {training_etc_model_time:.4f} seconds")
print(f"Test (Inference) time Extra Trees Classifier Model \t: {test_etc_model_time:.4f} seconds")

print()
print('Score AUPRC Extra Trees Classifier Model \t:',  etc_v1_auprc)

Training time Extra Trees Classifier Model 	: 30.5728 seconds
Test (Inference) time Extra Trees Classifier Model 	: 0.7454 seconds

Score AUPRC Extra Trees Classifier Model 	: 0.8140471847549451


In [79]:
importance_etc = pd.DataFrame({
    "feature": X_train.columns,
    "importance": etc_model.feature_importances_
}).sort_values("importance",ascending=False).reset_index(drop=True)

## Random Forest Classifier

In [80]:
start_rfc_train = time.perf_counter()

rfc_model.fit(X_train, y_train)

end_rfc_train = time.perf_counter()

training_rfc_model_time = end_rfc_train - start_rfc_train

In [81]:
start_rfc_test = time.perf_counter()

rfc_pred = rfc_model.predict_proba(X_test)[:, 1]

end_rfc_test = time.perf_counter()

test_rfc_model_time = end_rfc_test - start_rfc_test

In [82]:
rfc_v1_auprc = average_precision_score(y_test, rfc_pred)

In [83]:
print(f"Training time Random Forest Classifier Model: {training_rfc_model_time:.4f} seconds")
print(f"Test (Inference) time Random Forest Classifier Model: {test_rfc_model_time:.4f} seconds")

print()
print('Score AUPRC Random Forest Classifier Model \t:',  rfc_v1_auprc)

Training time Random Forest Classifier Model: 226.4055 seconds
Test (Inference) time Random Forest Classifier Model: 0.4622 seconds

Score AUPRC Random Forest Classifier Model 	: 0.8253427025737452


In [84]:
importance_rfc = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rfc_model.feature_importances_
}).sort_values("importance",ascending=False).reset_index(drop=True)

## CatBoost Classifier

In [85]:
start_cb_train = time.perf_counter()

cb_model.fit(X_train, y_train)

end_cb_train = time.perf_counter()

training_cb_model_time = end_cb_train - start_cb_train

In [86]:
start_cb_test = time.perf_counter()

cb_pred = cb_model.predict_proba(X_test)[:, 1]

end_cb_test = time.perf_counter()

test_cb_model_time = end_cb_test - start_cb_test

In [87]:
cb_v1_auprc = average_precision_score(y_test, cb_pred)

In [88]:
print(f"Training time CatBoost Classifier Model: {training_cb_model_time:.4f} seconds")
print(f"Test (Inference) time CatBoost Classifier Model: {test_cb_model_time:.4f} seconds")

print()
print('Score AUPRC CatBoost Classifier Model \t:',  cb_v1_auprc)

Training time CatBoost Classifier Model: 29.8928 seconds
Test (Inference) time CatBoost Classifier Model: 0.0538 seconds

Score AUPRC CatBoost Classifier Model 	: 0.8177673464031401


In [89]:
importance_cb = pd.DataFrame({
    "feature": X_train.columns,
    "importance": cb_model.get_feature_importance()
}).sort_values("importance",ascending=False).reset_index(drop=True)

# Feature Importance

## Mutual Information

In [90]:
mi = mutual_info_classif(X_train, y_train, random_state=42)

In [91]:
mi_df = pd.DataFrame({
    "features" : X_train.columns,
    "mi" : mi
})

In [92]:
mi_df.sort_values(by="mi", ascending=False)

,features,mi
17,V17,0.008013
14,V14,0.007946
10,V10,0.007320
12,V12,0.007315
11,V11,0.006577
16,V16,0.005876
3,V3,0.004671
4,V4,0.004623
9,V9,0.004051
18,V18,0.003992


## Model Importance

### Extra Trees Classifier

In [93]:
importance_etc

,feature,importance
0,V14,0.163281
1,V17,0.125352
2,V12,0.115412
3,V16,0.086488
4,V11,0.081330
5,V10,0.065061
6,V4,0.056778
7,V3,0.049992
8,V18,0.042136
9,V9,0.027910


### Random Forest Classifier

In [94]:
importance_rfc

,feature,importance
0,V14,0.172255
1,V10,0.124021
2,V12,0.106448
3,V4,0.096142
4,V17,0.087129
5,V11,0.068148
6,V3,0.054841
7,V16,0.034127
8,V7,0.026238
9,V2,0.022002


### CatBoost Classifier

In [95]:
importance_cb

,feature,importance
0,V1,8.957540
1,V4,7.406007
2,V26,6.904235
3,V14,6.164592
4,Amount,4.866981
5,V12,4.061221
6,V24,3.662033
7,V22,3.444703
8,V8,3.368629
9,V7,3.161832


## Summary Feature Importance

Dari hasil pengecekan top 10 feature importance dari 4 metode berbeda, dihasilkan sebagai berikut:
- Mutual Importance          = [v17, v14, v10, v12. v11, v16, v3, v4, v9, v18]
- Extra Trees Classifier     = [v14, v17, v12, v16. v11, v10, v4, v3, v18, v9]
- Random Forest Classifier   = [v14, v10, v12, v4. v17, v11, v3, v16, v7, v12]
- Catboost Classifier        = [v1, v4, v26, v14. Amount, v12, v24, v22, v8, v7]

Maka dilakukan penggabungan dari keempat nya menggunakan konsep Union sehingga hipotesa tercipta, bisa dilakukan reduksi feature sehingga hanya menggunakan feature [v17, v14, v10, v12, v11, v16, v3, v4, v9, v18, v7, v1, v26, Amount, v24, v22, v8]

# Export Model

In [96]:
cb_model.save_model("../../Output/Model/model_v1_catboost.cbm")
joblib.dump(etc_model,"../../Output/Model/model_v1_extra_trees.pkl")
joblib.dump(rfc_model,"../../Output/Model/model_v1_random_forest.pkl")

['../../Output/Model/model_v1_random_forest.pkl']